# Model 3 — Bidirectional LSTM

**Efficiency strategy:** Preload the full normalized (N, 301, 3) dataset into GPU VRAM once. Batch size raised to 1024 (was 256) — reduces train batches from ~1,372 to ~343 per epoch.

**Architecture:**
```
Input (batch, 301, 3)
BiLSTM(input=3, hidden=128, layers=2, dropout=0.2)
  → h_n[-2] (forward last layer)  | shape (batch, 128)
  → h_n[-1] (backward last layer) | shape (batch, 128)
  → concat → (batch, 256)
Linear(256→64) → ReLU → Dropout(0.3) → Linear(64→1)
```

**Training note:** Gradient clipping at `max_norm=1.0` is applied to stabilize training over 301 timesteps.

**Data:** `data/processed/sequences.h5` — battery-level split: 22 train / 6 test batteries.

In [ ]:
import h5py
import json
import math
import os
import sys
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.dataset import BatterySOHDataset, DEFAULT_TEST_BATTERIES
from src.models.lstm import SOHLSTM

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
H5_PATH      = "../data/processed/sequences.h5"
STATS_PATH   = "../data/processed/norm_stats.json"
CKPT_PATH    = "../results/checkpoints/lstm_best.pt"
METRICS_PATH = "../results/metrics/lstm_history.json"
FIG_DIR      = "../results/figures"

EPOCHS       = 100
BATCH_SIZE   = 1024   # raised from 256 — full dataset preloaded in VRAM
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 10
CLIP_GRAD    = 1.0    # gradient clipping — critical for LSTM over 301 timesteps
RANDOM_SEED  = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
os.makedirs(os.path.dirname(METRICS_PATH), exist_ok=True)

def set_seeds(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seeds()

## Section 1 — Data Loading + GPU Preload

In [ ]:
train_idx, test_idx = BatterySOHDataset.create_splits(H5_PATH)

if not os.path.exists(STATS_PATH):
    tmp = BatterySOHDataset(H5_PATH, indices=train_idx, normalize=False)
    tmp.compute_normalization_stats(STATS_PATH)

with open(STATS_PATH) as f:
    stats = json.load(f)
mean = np.array(stats['mean'], dtype=np.float32)
std  = np.array(stats['std'],  dtype=np.float32)

print('Loading dataset into RAM...', flush=True)
with h5py.File(H5_PATH, 'r') as f:
    X_all = f['X'][:]
    y_all = f['y'][:] / 100.0
print(f'  X: {X_all.shape}  ({X_all.nbytes / 1024**3:.2f} GB)')

X_all = (X_all - mean) / (std + 1e-8)

print('Pushing to GPU...', end=' ', flush=True)
X_tensor = torch.from_numpy(X_all).to(DEVICE)
y_tensor = torch.from_numpy(y_all).to(DEVICE)
del X_all, y_all
torch.cuda.empty_cache()
print(f'done. VRAM: {torch.cuda.memory_allocated() / 1024**2:.1f} MiB')

train_idx_t = torch.from_numpy(train_idx)
test_idx_t  = torch.from_numpy(test_idx)

X_train = X_tensor[train_idx_t]
y_train = y_tensor[train_idx_t]
X_test  = X_tensor[test_idx_t]
y_test  = y_tensor[test_idx_t]
del X_tensor, y_tensor

print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')
print(f'Test batteries: {DEFAULT_TEST_BATTERIES}')
print(f'VRAM after split: {torch.cuda.memory_allocated() / 1024**2:.1f} MiB')

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=0)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)

print(f'Train batches/epoch: {len(train_loader)}  (was 1372 with batch=256 + HDF5)')

## Section 2 — Gradient Clipping Analysis

LSTMs over long sequences (301 steps) can suffer from exploding gradients. This cell demonstrates why clipping is needed by logging the gradient norm before and after clipping during the first few steps.

In [ ]:
set_seeds()
_probe_model = SOHLSTM().to(DEVICE)
_probe_opt   = torch.optim.Adam(_probe_model.parameters(), lr=LR)
_criterion   = nn.MSELoss()

grad_norms_before = []
grad_norms_after  = []

_probe_model.train()
for i, (x, y) in enumerate(train_loader):
    if i >= 20:
        break
    _probe_opt.zero_grad(set_to_none=True)
    loss = _criterion(_probe_model(x).squeeze(1), y)
    loss.backward()

    norm_before = sum(p.grad.norm().item()**2 for p in _probe_model.parameters()
                      if p.grad is not None) ** 0.5
    nn.utils.clip_grad_norm_(_probe_model.parameters(), CLIP_GRAD)
    norm_after  = sum(p.grad.norm().item()**2 for p in _probe_model.parameters()
                      if p.grad is not None) ** 0.5

    grad_norms_before.append(norm_before)
    grad_norms_after.append(norm_after)
    _probe_opt.step()

del _probe_model, _probe_opt

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(grad_norms_before, label="Before clipping", color="crimson",    lw=1.5)
ax.plot(grad_norms_after,  label="After clipping",  color="steelblue",  lw=1.5)
ax.axhline(CLIP_GRAD, color="gray", ls="--", lw=1, label=f"Clip threshold ({CLIP_GRAD})")
ax.set_xlabel("Training step"); ax.set_ylabel("Gradient L2 norm")
ax.set_title("Gradient Norm: Before vs After Clipping (first 20 steps)")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lstm_grad_norms.png"), dpi=150)
plt.show()
print(f"Max grad norm before clipping: {max(grad_norms_before):.4f}")

## Section 3 — Model Architecture

In [ ]:
set_seeds()
model = SOHLSTM().to(DEVICE)
print(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {n_params:,}")

dummy = torch.randn(4, 301, 3).to(DEVICE)
with torch.no_grad():
    _, (h_n, _) = model.lstm(dummy)
    h_fwd = h_n[-2]
    h_bwd = h_n[-1]
    h_last = torch.cat([h_fwd, h_bwd], dim=1)
    out = model(dummy)

print(f"\nInput:                  {tuple(dummy.shape)}")
print(f"h_n (layers×dirs, B, H): {tuple(h_n.shape)}")
print(f"h_fwd (last fwd layer): {tuple(h_fwd.shape)}")
print(f"h_bwd (last bwd layer): {tuple(h_bwd.shape)}")
print(f"Concatenated hidden:    {tuple(h_last.shape)}")
print(f"Output:                 {tuple(out.shape)}")

## Section 4 — Training

In [ ]:
def train_epoch(model, loader, optimizer, criterion, clip_grad=CLIP_GRAD):
    model.train()
    total = 0.0
    bar = tqdm(loader, desc='Training', leave=True, unit='batch')
    for x, y in bar:
        # Data already on GPU
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x).squeeze(1), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)
        optimizer.step()
        total += loss.item()
        bar.set_postfix(loss=f'{loss.item():.4f}')
    return total / len(loader)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    bar = tqdm(loader, desc='Validating', leave=True, unit='batch')
    for x, y in bar:
        preds.append(model(x).squeeze(1).cpu())
        targets.append(y.cpu())
    preds   = torch.cat(preds).numpy() * 100.0
    targets = torch.cat(targets).numpy() * 100.0
    mae  = float(np.abs(preds - targets).mean())
    rmse = float(math.sqrt(((preds - targets)**2).mean()))
    return mae, rmse, preds, targets

In [ ]:
set_seeds()
model = SOHLSTM().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, factor=0.5)
criterion = nn.MSELoss()

history    = {"train_loss": [], "val_mae": [], "val_rmse": []}
best_val_mae = float("inf")
no_improve   = 0

print(f'Batch size: {BATCH_SIZE}  |  Train batches/epoch: {len(train_loader)}')
print(f'VRAM before training: {torch.cuda.memory_allocated() / 1024**2:.1f} MiB')

for epoch in range(1, EPOCHS + 1):
    print(f'\n{"="*60}')
    print(f'Epoch {epoch}/{EPOCHS}')
    print(f'{"="*60}')

    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_mae, val_rmse, _, _ = evaluate(model, test_loader)
    scheduler.step(val_mae)

    history["train_loss"].append(train_loss)
    history["val_mae"].append(val_mae)
    history["val_rmse"].append(val_rmse)

    lr_now = optimizer.param_groups[0]['lr']
    vram   = torch.cuda.memory_allocated() / 1024**2

    print(f'\nTrain Loss: {train_loss:.4f}')
    print(f'Val MAE:    {val_mae:.4f}%')
    print(f'Val RMSE:   {val_rmse:.4f}%')
    print(f'LR:         {lr_now:.6f}')
    print(f'VRAM:       {vram:.0f} MiB')

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        no_improve   = 0
        torch.save({"epoch": epoch, "val_mae": val_mae,
                    "model_state_dict": model.state_dict()}, CKPT_PATH)
        print(f'Model saved to {CKPT_PATH}')
        print('✓ New best model saved!')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch}.')
            break

with open(METRICS_PATH, "w") as f:
    json.dump(history, f, indent=2)
print(f'\nBest val MAE: {best_val_mae:.4f}%  |  Checkpoint: {CKPT_PATH}')

## Section 5 — Results

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val MAE {ckpt['val_mae']:.4f}%)")

test_mae, test_rmse, preds, targets = evaluate(model, test_loader)
print(f"\nTest MAE:  {test_mae:.4f} SOH%")
print(f"Test RMSE: {test_rmse:.4f} SOH%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history["train_loss"], color="#55A868", lw=1.5)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Training Loss"); axes[0].grid(alpha=0.3)

axes[1].plot(history["val_mae"],  label="Val MAE",  color="darkorange", lw=1.5)
axes[1].plot(history["val_rmse"], label="Val RMSE", color="crimson",    lw=1.5, ls="--")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("SOH%")
axes[1].set_title("Validation Metrics"); axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle("Bidirectional LSTM — Training Curves", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lstm_training_curves.png"), dpi=150)
plt.show()

In [ ]:
errors = preds - targets
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

lims = [min(targets.min(), preds.min()) - 2, max(targets.max(), preds.max()) + 2]
axes[0].scatter(targets, preds, alpha=0.25, s=5, color="#55A868")
axes[0].plot(lims, lims, "k--", lw=1)
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel("Actual SOH (%)"); axes[0].set_ylabel("Predicted SOH (%)")
axes[0].set_title(f"Predicted vs Actual  MAE={test_mae:.2f}%  RMSE={test_rmse:.2f}%")
axes[0].grid(alpha=0.3)

axes[1].hist(errors, bins=60, color="#55A868", edgecolor="white", alpha=0.85)
axes[1].axvline(0,              color="black", lw=1.2, ls="--", label="Zero error")
axes[1].axvline(errors.mean(),  color="red",   lw=1.5, label=f"Mean={errors.mean():.2f}%")
axes[1].set_xlabel("Prediction Error (SOH%)"); axes[1].set_ylabel("Count")
axes[1].set_title(f"Error Distribution  std={errors.std():.2f}%")
axes[1].legend(); axes[1].grid(axis="y", alpha=0.3)

fig.suptitle("Bidirectional LSTM — Test Set Results", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lstm_results.png"), dpi=150)
plt.show()

In [ ]:
# Visualise hidden state activation magnitudes at sequence end
model.eval()

with torch.no_grad():
    outputs, (h_n, _) = model.lstm(X_test[:BATCH_SIZE])
    h_fwd = h_n[-2].cpu().numpy()   # (batch, 128) last layer forward

soh_vals = y_test[:BATCH_SIZE].cpu()
sort_idx = soh_vals.argsort().numpy()
h_sorted   = h_fwd[sort_idx]
soh_sorted = soh_vals[sort_idx].numpy() * 100

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(h_sorted.T, aspect="auto", cmap="RdBu_r",
               vmin=-2, vmax=2, interpolation="nearest")
ax.set_xlabel(f"Sample (sorted by SOH, {soh_sorted[0]:.1f}% → {soh_sorted[-1]:.1f}%)")
ax.set_ylabel("Hidden unit index (128 forward units)")
ax.set_title("LSTM Hidden State Magnitude at Sequence End — Forward Direction")
plt.colorbar(im, ax=ax, label="Activation")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lstm_hidden_states.png"), dpi=150)
plt.show()

In [ ]:
print("=" * 40)
print("BIDIRECTIONAL LSTM — FINAL SUMMARY")
print("=" * 40)
print(f"Parameters : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Batch size : {BATCH_SIZE}  (batches/epoch: {len(train_loader)})")
print(f"Test MAE   : {test_mae:.4f} SOH%")
print(f"Test RMSE  : {test_rmse:.4f} SOH%")
print(f"Error mean : {errors.mean():.4f} SOH%")
print(f"Error std  : {errors.std():.4f} SOH%")
print(f"Error p95  : {np.percentile(np.abs(errors), 95):.4f} SOH%")